# Testing phase 1 functions

In [1]:
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
from PIL import Image
from phase1_setup import parse_weight, make_portion_id, is_blurry, validate_image, assign_splits
import cv2

## Test portions dict creation

In [2]:
dataset_dir = Path('/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets')
raw_dir = dataset_dir / 'raw'

In [3]:
food_type_dirs = sorted([d for d in raw_dir.iterdir() if d.is_dir()])
print(len(food_type_dirs))
print(food_type_dirs[1])

10
/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Gnocchi


In [ ]:

portions = []
skipped_folders = []
food_type_counters = {}


for food_type_dir in food_type_dirs:
    food_type = food_type_dir.name
    food_type_counters[food_type] = 0
    # print(food_type)
    portion_dirs = sorted([d for d in food_type_dir.iterdir() if d.is_dir()])
    # print(portion_dirs)
    
    for portion_dir in portion_dirs:
        weight = parse_weight(portion_dir.name)    

        if weight is None:
            print(f"Skipping {portion_dir} due to unparseable weight.")
            skipped_folders.append(portion_dir)
            continue
        
        food_type_counters[food_type] += 1
        portion_id = make_portion_id(food_type, food_type_counters[food_type])   
        portions.append({
            "portion_id": portion_id,
            "food_type": food_type,
            "weight": weight,
            "originnal_folder": portion_dir
        })

portion = portions[3]
print(portion)

{'portion_id': 'cananderli_004', 'food_type': 'Cananderli', 'weight': 250.0, 'originnal_folder': PosixPath('/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Cananderli/250g')}


## Test the dataset split

In [15]:
all_ids = [p['portion_id'] for p in portions]
all_types = [p['food_type'] for p in portions]
all_dirs = [p['originnal_folder'] for p in portions]
all_weights = [p['weight'] for p in portions]

print(all_ids[9:13])
print(all_types[9:13])

['cananderli_010', 'gnocchi_001', 'gnocchi_002', 'gnocchi_003']
['Cananderli', 'Gnocchi', 'Gnocchi', 'Gnocchi']


In [ ]:
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42


x = assign_splits(all_ids, all_types)
print(x)


In [ ]:
df = pd.DataFrame(
    {
        "portion_id": all_ids,
        "food_type": all_types,
        "dir": all_dirs,
        "weight": all_weights
    }
)

df["split"] = df["portion_id"].map(x)
df.head(10)

,portion_id,food_type,dir,weight
0,cananderli_001,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,100.0
1,cananderli_002,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,150.0
2,cananderli_003,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,200.0
3,cananderli_004,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,250.0
4,cananderli_005,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,25.0


In [ ]:
for portion in portions:
    print(portion)